In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
players = pd.read_csv("../data/raw/kaggle/players.csv")
teams = pd.read_csv("../data/raw/kaggle/teams.csv")
matches = pd.read_csv("../data/raw/kaggle/matches.csv")

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


In [3]:
players_clean = players.copy()
teams_clean = teams.copy()
matches_clean = matches.copy()

print("Working on copied datasets")

Working on copied datasets


In [4]:
def clean_columns(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("%", "pct")
        .str.replace("/", "_")
        .str.replace("-", "_")
    )

    return df

players_clean = clean_columns(players_clean)
teams_clean = clean_columns(teams_clean)
matches_clean = clean_columns(matches_clean)

print("Column names standardized")

Column names standardized


In [6]:
print("Players duplicates :", players_clean.duplicated().sum())
print("Teams duplicates :", teams_clean.duplicated().sum())
print("Matches duplicates :", matches_clean.duplicated().sum())

players_clean.drop_duplicates(inplace=True)
teams_clean.drop_duplicates(inplace=True)
matches_clean.drop_duplicates(inplace=True)

print("Duplicate removal complete")

Players duplicates : 0
Teams duplicates : 0
Matches duplicates : 0
Duplicate removal complete


In [7]:
players_clean = players_clean.dropna(axis=1, how="all")
teams_clean = teams_clean.dropna(axis=1, how="all")
matches_clean = matches_clean.dropna(axis=1, how="all")

print("Removed completely empty columns")

Removed completely empty columns


In [8]:
columns_to_remove = [
    "pens_won",
    "pens_conceded"
]

for col in columns_to_remove:

    if col in players_clean.columns:

        players_clean.drop(columns=col, inplace=True)

print("Removed useless player columns")

Removed useless player columns


In [9]:
players_clean["club"] = players_clean["club"].fillna("Unknown Club")

print("Missing club values filled")

Missing club values filled


In [10]:
matches_clean["date"] = pd.to_datetime(
    matches_clean["date"],
    errors="coerce"
)

print("Date converted")

Date converted


In [11]:
matches_clean["attendance"] = (
    matches_clean["attendance"]
    .astype(str)
    .str.replace(",", "")
)

matches_clean["attendance"] = pd.to_numeric(
    matches_clean["attendance"],
    errors="coerce"
)

print("Attendance cleaned")

Attendance cleaned


In [12]:
numeric_players = players_clean.select_dtypes(include=["number"]).columns

players_clean[numeric_players] = (
    players_clean[numeric_players]
    .fillna(0)
)

numeric_teams = teams_clean.select_dtypes(include=["number"]).columns

teams_clean[numeric_teams] = (
    teams_clean[numeric_teams]
    .fillna(0)
)

numeric_matches = matches_clean.select_dtypes(include=["number"]).columns

matches_clean[numeric_matches] = (
    matches_clean[numeric_matches]
    .fillna(0)
)

print("Numeric missing values handled")


Numeric missing values handled


In [13]:
object_players = players_clean.select_dtypes(include="object").columns

players_clean[object_players] = (
    players_clean[object_players]
    .fillna("Unknown")
)

object_teams = teams_clean.select_dtypes(include="object").columns

teams_clean[object_teams] = (
    teams_clean[object_teams]
    .fillna("Unknown")
)

object_matches = matches_clean.select_dtypes(include="object").columns

matches_clean[object_matches] = (
    matches_clean[object_matches]
    .fillna("Unknown")
)

print("Text missing values handled")

Text missing values handled


C:\Users\AYUSH SINGH\AppData\Local\Temp\ipykernel_21716\1116165709.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_players = players_clean.select_dtypes(include="object").columns
C:\Users\AYUSH SINGH\AppData\Local\Temp\ipykernel_21716\1116165709.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas

In [17]:
goalkeepers = players_clean[
    players_clean["position"].str.contains(
        "GK",
        case=False,
        na=False
    )
]

outfield_players = players_clean[
    ~players_clean["position"].str.contains(
        "GK",
        case=False,
        na=False
    )
]

print("Goalkeepers :", len(goalkeepers))
print("Outfield Players :", len(outfield_players))

Goalkeepers : 145
Outfield Players : 1103


In [18]:
summary = pd.DataFrame({

    "Dataset":[
        "Players",
        "Outfield",
        "Goalkeepers",
        "Teams",
        "Matches"
    ],

    "Rows":[
        len(players_clean),
        len(outfield_players),
        len(goalkeepers),
        len(teams_clean),
        len(matches_clean)
    ],

    "Columns":[
        players_clean.shape[1],
        outfield_players.shape[1],
        goalkeepers.shape[1],
        teams_clean.shape[1],
        matches_clean.shape[1]
    ]
})

summary

,Dataset,Rows,Columns
0,Players,1248,70
1,Outfield,1103,70
2,Goalkeepers,145,70
3,Teams,48,132
4,Matches,104,42


In [19]:
players_clean.to_csv(
    "../data/interim/players_clean.csv",
    index=False
)

teams_clean.to_csv(
    "../data/interim/teams_clean.csv",
    index=False
)

matches_clean.to_csv(
    "../data/interim/matches_clean.csv",
    index=False
)

goalkeepers.to_csv(
    "../data/interim/goalkeepers.csv",
    index=False
)

outfield_players.to_csv(
    "../data/interim/outfield_players.csv",
    index=False
)

summary.to_csv(
    "../reports/cleaning_summary.csv",
    index=False
)

print("=" * 60)
print("Cleaning Completed Successfully")
print("=" * 60)

print("Saved:")
print("✓ players_clean.csv")
print("✓ teams_clean.csv")
print("✓ matches_clean.csv")
print("✓ goalkeepers.csv")
print("✓ outfield_players.csv")
print("✓ cleaning_summary.csv")

Cleaning Completed Successfully
Saved:
✓ players_clean.csv
✓ teams_clean.csv
✓ matches_clean.csv
✓ goalkeepers.csv
✓ outfield_players.csv
✓ cleaning_summary.csv
